# Concurrency Stress Tests

## Test Case
- **Users**: 50 concurrent users  
- **Duration**: 30-min mixed R/W workload  
- **Target**: P99 latency stability under heavy load

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Create dedicated table for concurrency testing
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CONCURRENCY_TEST (
    id INT,
    thread_id INT,
    operation STRING,
    payload VARIANT,
    created_at TIMESTAMP_NTZ(9)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

-- Pre-populate with baseline data
INSERT INTO ICEBERG_POC.TESTS.CONCURRENCY_TEST
SELECT 
    SEQ4() AS id,
    0 AS thread_id,
    'baseline' AS operation,
    OBJECT_CONSTRUCT('data', 'initial_' || SEQ4()) AS payload,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

## Concurrent Read Queries
Run these queries simultaneously from multiple sessions to test read concurrency.

In [ ]:
-- Read Query 1: Full scan aggregation
SELECT operation, COUNT(*), AVG(id) FROM ICEBERG_POC.TESTS.CONCURRENCY_TEST GROUP BY 1;

-- Read Query 2: Point lookup
SELECT * FROM ICEBERG_POC.TESTS.CONCURRENCY_TEST WHERE id = 50000;

-- Read Query 3: Range scan
SELECT COUNT(*) FROM ICEBERG_POC.TESTS.CONCURRENCY_TEST WHERE id BETWEEN 10000 AND 20000;

-- Read Query 4: VARIANT query
SELECT id, payload:data::STRING FROM ICEBERG_POC.TESTS.CONCURRENCY_TEST WHERE id < 100;

## Concurrent Write Operations
Test mixed read/write workloads to measure P99 latency stability.

In [ ]:
-- Write Operation 1: Insert batch
INSERT INTO ICEBERG_POC.TESTS.CONCURRENCY_TEST
SELECT 
    (SELECT MAX(id) FROM ICEBERG_POC.TESTS.CONCURRENCY_TEST) + SEQ4() AS id,
    1 AS thread_id,
    'concurrent_insert' AS operation,
    OBJECT_CONSTRUCT('batch', 1, 'timestamp', CURRENT_TIMESTAMP()) AS payload,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Write Operation 2: Update batch
UPDATE ICEBERG_POC.TESTS.CONCURRENCY_TEST 
SET payload = OBJECT_INSERT(payload, 'updated', TRUE),
    operation = 'concurrent_update'
WHERE id BETWEEN 1000 AND 2000;

## Query History Analysis
Analyze query latencies from concurrent workload.

In [ ]:
-- Analyze query performance for Iceberg table operations
SELECT 
    QUERY_TYPE,
    COUNT(*) AS query_count,
    AVG(TOTAL_ELAPSED_TIME) / 1000 AS avg_latency_sec,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000 AS p95_latency_sec,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000 AS p99_latency_sec,
    MAX(TOTAL_ELAPSED_TIME) / 1000 AS max_latency_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_TEXT ILIKE '%CONCURRENCY_TEST%'
    AND START_TIME > DATEADD('hour', -1, CURRENT_TIMESTAMP())
GROUP BY QUERY_TYPE
ORDER BY query_count DESC;